# Resultados del pipeline

Este notebook **no calcula nada nuevo**: solo lee lo que dejó `python -m dengue_alerta todo` y arma las tablas y figuras para la tesis.

Regla del proyecto: *la lógica vive en `src/`, los notebooks solo exploran y presentan.*

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METRICAS = RAIZ / "reports" / "metricas"
resultados = pd.read_csv(METRICAS / "resultados.csv")
resultados.head()

## 1. Resumen del panel y auditoría de fuga

In [ ]:
print(json.dumps(json.loads((METRICAS / "resumen_panel.json").read_text()), indent=2, ensure_ascii=False))
print(json.dumps(json.loads((METRICAS / "auditoria_fuga.json").read_text()), indent=2, ensure_ascii=False))

## 2. Tabla de ablación (PR-AUC en prueba)

La columna clave es **sin_brote_hoy**: mide la capacidad de anticipar brotes *nuevos*, donde la persistencia no sirve.

In [ ]:
tabla = resultados.pivot_table(index=["experimento", "modelo"], columns=["subgrupo", "horizonte"], values="pr_auc").round(3)
tabla

## 3. Figura: aporte de cada grupo de variables

In [ ]:
sub = resultados[(resultados.subgrupo == "sin_brote_hoy") & (resultados.experimento != "linea_base")]
mejor = sub.groupby(["experimento", "horizonte"]).pr_auc.max().unstack()
base = resultados[(resultados.subgrupo == "sin_brote_hoy") & (resultados.modelo == "persistencia")].set_index("horizonte").pr_auc
ax = mejor.plot.bar(figsize=(7, 4), rot=0)
for h, v in base.items():
    ax.axhline(v, ls="--", lw=1, color="gray")
ax.set_ylabel("PR-AUC (distritos sin brote hoy)")
ax.set_title("Ablación: cada grupo de variables suma información")
ax.legend(title="Horizonte (semanas)")
plt.tight_layout()

## 4. Variables más influyentes (SHAP)

In [ ]:
imp = pd.read_csv(METRICAS / "shap_importancia_h2.csv").head(10)
imp